In [0]:
%pip install openai
dbutils.library.restartPython()

In [0]:
# UC4 — AI Business Insights (COMPLETE)
# Reads : All primeinsurance_analytics.gold tables
# Writes: primeinsurance_analytics.gold.ai_business_insights
#
# HOW TO IMPORT INTO DATABRICKS:
#   Workspace → Import → select this .py file → Import
#   Then attach to a cluster and run cell by cell

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Imports and LLM Client Setup
# ─────────────────────────────────────────────────────────────

import json
import uuid
from openai import OpenAI
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    FloatType, BooleanType, TimestampType, IntegerType
)

# Auto-fetch Databricks token — no manual key needed
DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils().notebook().getContext()
    .apiToken().get()
)

WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")

client = OpenAI(
    api_key  = DATABRICKS_TOKEN,
    base_url = f"https://{WORKSPACE_URL}/serving-endpoints"
)

MODEL = "databricks-gpt-oss-20b"

print("✅ Client ready")
print(f"   Workspace : {WORKSPACE_URL}")
print(f"   Model     : {MODEL}")

In [0]:
import json, ast

def get_llm_response(prompt: str, system: str = None, max_tokens: int = 800) -> str:

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model      = MODEL,
        messages   = messages,
        max_tokens = max_tokens
    )

    raw = response.choices[0].message.content

    # ── Case 1: raw is already a Python list ──────────────
    if isinstance(raw, list):
        for block in raw:
            if isinstance(block, dict) and block.get("type") == "text":
                return block.get("text", "").strip()
        # if no "text" block found, look inside nested "summary"
        for block in raw:
            if isinstance(block, dict):
                for sub in block.get("summary", []):
                    if isinstance(sub, dict) and sub.get("type") == "summary_text":
                        return sub.get("text", "").strip()

    # ── Case 2: raw is a string ────────────────────────────
    if isinstance(raw, str):

        # Try JSON parse first
        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                for block in parsed:
                    if isinstance(block, dict) and block.get("type") == "text":
                        return block.get("text", "").strip()
        except (json.JSONDecodeError, TypeError):
            pass

        # Try Python literal parse (handles single-quoted list strings)
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                for block in parsed:
                    if isinstance(block, dict) and block.get("type") == "text":
                        return block.get("text", "").strip()
        except (ValueError, SyntaxError):
            pass

        # Plain string — return directly
        return raw.strip()

    return str(raw).strip()


# Quick test — prints what the model actually returns
test = get_llm_response("Say hello in one word.")
print("TEST RESPONSE:", repr(test))

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Collect KPIs from All Gold Tables
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F

print("📊 Collecting KPIs from all gold tables...\n")

# ── KPI Block 1: fact_claims ──────────────────────────────────

claims_kpi = (
    spark.read.table("primeinsurance_analytics.gold.fact_claims")
    .agg(
        F.count("claim_id").alias("total_claims"),
        F.round(F.avg("total_claim_amount"), 2).alias("avg_claim_amount"),
        F.round(F.sum("total_claim_amount"), 2).alias("total_claimed"),
        F.sum(F.col("is_rejected").cast("int")).alias("total_rejected"),
        F.round(F.avg("days_to_process"), 1).alias("avg_days_to_process"),
        F.countDistinct("customer_id").alias("unique_customers_with_claims"),
        F.round(F.max("total_claim_amount"), 2).alias("largest_single_claim"),
        # ✅ NEW — now meaningful with real dates
        F.sum(F.when(F.col("claim_status") == "open", 1).otherwise(0))
         .alias("open_claims"),
        F.sum(F.when(F.col("claim_status") == "closed", 1).otherwise(0))
         .alias("closed_claims"),
    )
    .collect()[0].asDict()
)

rejection_rate = round(
    (claims_kpi["total_rejected"] / claims_kpi["total_claims"]) * 100, 2
) if claims_kpi["total_claims"] > 0 else 0.0

print(f"✅ fact_claims: {claims_kpi['total_claims']:,} claims, {rejection_rate}% rejection rate")

# ── KPI Block 2: fact_sales ───────────────────────────────────
sales_kpi = (
    spark.read.table("primeinsurance_analytics.gold.fact_sales")
    .agg(
        F.count("sales_id").alias("total_listings"),
        F.sum(F.when(F.col("is_sold") == True, 1).otherwise(0)).alias("total_sold"),
        F.sum(F.when(F.col("is_sold") == False, 1).otherwise(0)).alias("total_unsold"),
        F.round(F.avg("selling_price"), 2).alias("avg_selling_price"),
        F.round(F.sum(
            F.when(F.col("is_sold") == False, F.col("selling_price")).otherwise(0)
        ), 2).alias("total_unsold_value"),
        F.round(F.avg(F.when(F.col("is_sold") == True, F.col("days_to_sell"))), 1).alias("avg_days_to_sell_sold"),
        F.round(F.avg(F.when(F.col("is_sold") == False, F.col("days_to_sell"))), 1).alias("avg_days_listed_unsold")
    )
    .collect()[0].asDict()
)

sell_through_rate = round(
    (sales_kpi["total_sold"] / sales_kpi["total_listings"]) * 100, 2
) if sales_kpi["total_listings"] > 0 else 0.0

print(f"✅ fact_sales: {sales_kpi['total_listings']:,} listings, {sell_through_rate}% sell-through")

# ── KPI Block 3: Rejection by region ─────────────────────────
worst_region_row = (
    spark.read.table("primeinsurance_analytics.gold.agg_claim_rejection_by_region_month")
    .orderBy(F.col("rejection_rate_pct").desc())
    .limit(1)
    .collect()[0].asDict()
)

best_region_row = (
    spark.read.table("primeinsurance_analytics.gold.agg_claim_rejection_by_region_month")
    .orderBy(F.col("rejection_rate_pct").asc())
    .limit(1)
    .collect()[0].asDict()
)

region_summary = (
    spark.read.table("primeinsurance_analytics.gold.agg_claim_rejection_by_region_month")
    .groupBy("region")
    .agg(
        F.round(F.avg("rejection_rate_pct"), 2).alias("avg_rejection_rate"),
        F.sum("total_claims").alias("total_claims")
    )
    .orderBy("region")
    .collect()
)

region_summary_str = "\n".join([
    f"  {r['region']}: {r['avg_rejection_rate']}% rejection rate ({r['total_claims']:,} claims)"
    for r in region_summary
])
# ── KPI Block 4: Processing time by severity ──────────────────
severity_rows = (
    spark.read.table("primeinsurance_analytics.gold.agg_claim_processing_time_by_severity")
    .orderBy(F.col("avg_days_to_process").desc())
    .collect()
)

worst_severity   = severity_rows[0].asDict()  if severity_rows          else {}
fastest_severity = severity_rows[-1].asDict() if len(severity_rows) > 1 else {}

severity_summary_str = "\n".join([
    f"  {r['incident_severity']}: avg {r['avg_days_to_process']} days "
    f"(max {r['max_days_to_process']} days, {r['total_claims']:,} claims)"
    for r in severity_rows
])

print(f"✅ Severity KPIs: slowest={worst_severity.get('incident_severity', 'N/A')} "
      f"at {worst_severity.get('avg_days_to_process', 0)} days")

# ── KPI Block 5: Unsold inventory by manufacturer ─────────────
unsold_rows = (
    spark.read.table("primeinsurance_analytics.gold.agg_unsold_inventory_by_model_region")
    .limit(5)
    .collect()
)

top_unsold = unsold_rows[0].asDict() if unsold_rows else {}

unsold_dicts = [r.asDict() for r in unsold_rows] # convert once, use safely
unsold_summary_str = "\n".join([
f" {r.get('manufacturer', r.get('model', 'Unknown'))}: "
f"{r.get('unsold_listings', 0):,} unsold "
f"(${float(r.get('total_unsold_value', r.get('total_listed_value', 0))):,.2f} value, "
f"avg {r.get('avg_days_listed', 'N/A')} days listed)"
for r in unsold_dicts
])

print(f"✅ Unsold inventory: "
      f"{top_unsold.get('manufacturer', 'N/A')} leads with "
      f"{top_unsold.get('unsold_listings', 0):,} unsold")
# ── KPI Block 6: dim_customer ─────────────────────────────────
customer_kpi = (
    spark.read.table("primeinsurance_analytics.gold.dim_customer")
    .agg(
        F.count("customer_id").alias("total_customers"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
        F.sum(F.when(F.col("balance") < 0, 1).otherwise(0)).alias("negative_balance_count"),
        F.sum(F.col("has_home_insurance").cast("int")).alias("with_home_insurance"),
        F.sum(F.col("has_car_loan").cast("int")).alias("with_car_loan")
    )
    .collect()[0].asDict()
)

home_ins_pct = round(customer_kpi["with_home_insurance"] / customer_kpi["total_customers"] * 100, 1)
car_loan_pct = round(customer_kpi["with_car_loan"] / customer_kpi["total_customers"] * 100, 1)

print(f"✅ Customer KPIs: {customer_kpi['total_customers']:,} customers, avg balance ${float(customer_kpi['avg_balance']):,.2f}")

# ── KPI Block 7: dim_policy ───────────────────────────────────
policy_kpi = (
    spark.read.table("primeinsurance_analytics.gold.dim_policy")
    .agg(
        F.count("policy_number").alias("total_policies"),
        F.sum(F.col("is_active").cast("int")).alias("active_policies"),
        F.round(F.avg("annual_premium"), 2).alias("avg_premium"),
        F.round(F.sum("annual_premium"), 2).alias("total_premium_book"),
        F.round(F.avg("deductible"), 2).alias("avg_deductible"),
        F.round(F.avg("umbrella_limit"), 2).alias("avg_umbrella_limit"),
        F.sum(F.when(F.col("umbrella_limit") > 0, 1).otherwise(0)).alias("policies_with_umbrella"),
        F.round(F.avg("csl_bodily_injury_per_person"), 2).alias("avg_bi_per_person"),
        F.round(F.avg("csl_bodily_injury_per_accident"), 2).alias("avg_bi_per_accident"),
        F.countDistinct("policy_state").alias("states_covered")
    )
    .collect()[0].asDict()
)

active_pct      = round(policy_kpi["active_policies"] / policy_kpi["total_policies"] * 100, 1)
umbrella_pct    = round(policy_kpi["policies_with_umbrella"] / policy_kpi["total_policies"] * 100, 1)

# State distribution
state_dist = (
    spark.read.table("primeinsurance_analytics.gold.dim_policy")
    .groupBy("policy_state")
    .agg(F.count("policy_number").alias("policy_count"))
    .orderBy(F.col("policy_count").desc())
    .limit(5)
    .collect()
)

state_dist_str = ", ".join([f"{r['policy_state']} ({r['policy_count']:,})" for r in state_dist])

print(f"✅ Policy KPIs: {policy_kpi['total_policies']:,} policies, {active_pct}% active, {umbrella_pct}% with umbrella")

print("\n📊 All KPIs collected successfully")

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Domain 1: Policy Portfolio Insight
# Business domain: underwriting, coverage exposure, premium book
# ─────────────────────────────────────────────────────────────

def generate_policy_portfolio_insight() -> tuple:
    """
    Generates an executive summary for the policy portfolio domain.
    Returns (summary_text, kpi_dict) for auditability.
    """

    kpis = {
        "total_policies"        : int(policy_kpi["total_policies"]),
        "active_policies"       : int(policy_kpi["active_policies"]),
        "active_pct"            : active_pct,
        "avg_premium"           : float(policy_kpi["avg_premium"]),
        "total_premium_book"    : float(policy_kpi["total_premium_book"]),
        "avg_deductible"        : float(policy_kpi["avg_deductible"]),
        "avg_umbrella_limit"    : float(policy_kpi["avg_umbrella_limit"]),
        "policies_with_umbrella": int(policy_kpi["policies_with_umbrella"]),
        "umbrella_pct"          : umbrella_pct,
        "avg_bi_per_person"     : float(policy_kpi["avg_bi_per_person"]),
        "avg_bi_per_accident"   : float(policy_kpi["avg_bi_per_accident"]),
        "states_covered"        : int(policy_kpi["states_covered"]),
        "top_5_states_by_volume": state_dist_str
    }

    system = """You are the Chief Underwriting Officer at PrimeInsurance.
Write a concise policy portfolio briefing for the executive team.
Structure:

PORTFOLIO HEADLINE
One sentence — the single most important portfolio insight today.

KEY FINDINGS
Finding 1: [coverage concentration or exposure risk — cite a number]
Finding 2: [umbrella / high-limit exposure — cite a number]
Finding 3: [premium book health — cite a number]

RECOMMENDED ACTIONS
Action 1: [specific underwriting or pricing action]
Action 2: [specific risk management action]

RISK WATCH
One sentence — largest underwriting risk if unaddressed in 60 days.

Rules: Every finding must cite a specific number. No jargon. 4 short paragraphs max."""

    prompt = f"""Write a policy portfolio briefing based on this aggregated data:

PORTFOLIO OVERVIEW:
  Total Policies               : {kpis['total_policies']:,}
  Active Policies              : {kpis['active_policies']:,} ({kpis['active_pct']}% of portfolio)
  Inactive / Lapsed            : {kpis['total_policies'] - kpis['active_policies']:,}

PREMIUM & FINANCIAL EXPOSURE:
  Average Annual Premium       : ${kpis['avg_premium']:,.2f}
  Total Premium Book Value     : ${kpis['total_premium_book']:,.2f}
  Average Deductible           : ${kpis['avg_deductible']:,.2f}

COVERAGE LIMITS:
  Policies with Umbrella       : {kpis['policies_with_umbrella']:,} ({kpis['umbrella_pct']}% of portfolio)
  Average Umbrella Limit       : ${kpis['avg_umbrella_limit']:,.2f}
  Avg Bodily Injury per Person : ${kpis['avg_bi_per_person']:,.2f}
  Avg Bodily Injury per Accident: ${kpis['avg_bi_per_accident']:,.2f}

GEOGRAPHIC DISTRIBUTION:
  States Covered               : {kpis['states_covered']}
  Top 5 States by Policy Volume: {kpis['top_5_states_by_volume']}
"""

    return get_llm_response(prompt, system=system, max_tokens=700), kpis


print("🤖 Generating Domain 1: Policy Portfolio insight...")
policy_summary, policy_kpis_used = generate_policy_portfolio_insight()

print("\n" + "=" * 70)
print("DOMAIN 1 — POLICY PORTFOLIO")
print("=" * 70)
print(policy_summary)
print("=" * 70)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Domain 2: Claims Performance Insight
# Business domain: processing efficiency, rejection rates, backlog
# ─────────────────────────────────────────────────────────────

def generate_claims_performance_insight() -> tuple:
    """
    Generates an executive summary for the claims performance domain.
    Returns (summary_text, kpi_dict) for auditability.
    """

    kpis = {
        "total_claims"              : int(claims_kpi["total_claims"]),
        "total_rejected"            : int(claims_kpi["total_rejected"]),
        "rejection_rate_pct"        : float(rejection_rate),
        "total_claimed_amount"      : float(claims_kpi["total_claimed"]),
        "avg_claim_amount"          : float(claims_kpi["avg_claim_amount"]),
        "largest_single_claim"      : float(claims_kpi["largest_single_claim"]),
        "avg_days_to_process"       : float(claims_kpi["avg_days_to_process"]) if claims_kpi["avg_days_to_process"] is not None else 0.0,
        "unique_customers_w_claims" : int(claims_kpi["unique_customers_with_claims"]),
        "worst_region"              : str(worst_region_row.get("region", "")),
        "worst_region_rejection_pct": float(worst_region_row.get("rejection_rate_pct", 0)),
        "best_region"               : str(best_region_row.get("region", "")),
        "best_region_rejection_pct" : float(best_region_row.get("rejection_rate_pct", 0)),
        "slowest_severity"          : str(worst_severity.get("incident_severity", "")),
        "slowest_severity_days"     : float(worst_severity.get("avg_days_to_process", 0)) if worst_severity.get("avg_days_to_process") is not None else 0.0,
        "fastest_severity"          : str(fastest_severity.get("incident_severity", "")),
        "fastest_severity_days"     : float(fastest_severity.get("avg_days_to_process", 0)) if fastest_severity.get("avg_days_to_process") is not None else 0.0
    }

    system = """You are a claims operations consultant at an insurance company.
Write a sharp claims performance briefing for the VP of Claims Operations.
Structure:

CLAIMS HEADLINE
One sentence — the single most urgent claims issue.

KEY FINDINGS
Finding 1: [rejection rate insight — cite region and percentage]
Finding 2: [processing time bottleneck — cite severity and days]
Finding 3: [financial exposure — cite total amount and largest claim]

RECOMMENDED ACTIONS
Action 1: [specific operational fix with expected time saving]
Action 2: [specific regional intervention]

RISK WATCH
One sentence — regulatory or financial risk if processing delays continue.

Rules: Every finding must cite specific numbers. Be direct. No filler sentences."""

    avg_days_str = f"{kpis['avg_days_to_process']:.1f}" if kpis['avg_days_to_process'] > 0 else "N/A"

    prompt = f"""Write a claims performance briefing based on this aggregated data:

OVERALL CLAIMS PERFORMANCE:
  Total Claims Filed            : {kpis['total_claims']:,}
  Total Rejected                : {kpis['total_rejected']:,} ({kpis['rejection_rate_pct']}%)
  Total Amount Claimed          : ${kpis['total_claimed_amount']:,.2f}
  Average Claim Amount          : ${kpis['avg_claim_amount']:,.2f}
  Largest Single Claim          : ${kpis['largest_single_claim']:,.2f}
  Avg Processing Time           : {avg_days_str} days
  Unique Customers with Claims  : {kpis['unique_customers_w_claims']:,}

REJECTION RATE BY REGION:
{region_summary_str}
  Worst: {kpis['worst_region']} at {kpis['worst_region_rejection_pct']}%
  Best : {kpis['best_region']} at {kpis['best_region_rejection_pct']}%

PROCESSING TIME BY SEVERITY (slowest first):
{severity_summary_str}
  Slowest: {kpis['slowest_severity']} at {kpis['slowest_severity_days']:.1f} days avg
  Fastest: {kpis['fastest_severity']} at {kpis['fastest_severity_days']:.1f} days avg
"""

    return get_llm_response(prompt, system=system, max_tokens=700), kpis


print("🤖 Generating Domain 2: Claims Performance insight...")
claims_summary, claims_kpis_used = generate_claims_performance_insight()

print("\n" + "=" * 70)
print("DOMAIN 2 — CLAIMS PERFORMANCE")
print("=" * 70)
print(claims_summary)
print("=" * 70)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Domain 3: Customer Profile Insight
# Business domain: customer health, cross-sell opportunity, risk segments
# ─────────────────────────────────────────────────────────────

def generate_customer_profile_insight() -> tuple:
    """
    Generates an executive summary for the customer profile domain.
    Returns (summary_text, kpi_dict) for auditability.
    """

    kpis = {
        "total_customers"           : int(customer_kpi["total_customers"]),
        "avg_balance"               : float(customer_kpi["avg_balance"]),
        "negative_balance_count"    : int(customer_kpi["negative_balance_count"]),
        "negative_balance_pct"      : round(customer_kpi["negative_balance_count"] / customer_kpi["total_customers"] * 100, 1),
        "with_home_insurance"       : int(customer_kpi["with_home_insurance"]),
        "home_insurance_pct"        : home_ins_pct,
        "with_car_loan"             : int(customer_kpi["with_car_loan"]),
        "car_loan_pct"              : car_loan_pct,
        "unique_customers_w_claims" : int(claims_kpi["unique_customers_with_claims"]),
        "claims_penetration_pct"    : round(claims_kpi["unique_customers_with_claims"] / customer_kpi["total_customers"] * 100, 1)
    }

    system = """You are a customer analytics lead at an insurance company.
Write a sharp customer profile briefing for the Chief Marketing Officer.
Structure:

CUSTOMER HEADLINE
One sentence — the most important customer base insight.

KEY FINDINGS
Finding 1: [financial health of customer base — cite balance data]
Finding 2: [cross-sell opportunity — cite home insurance or car loan penetration]
Finding 3: [claims risk concentration — cite penetration rate]

RECOMMENDED ACTIONS
Action 1: [specific retention or upsell action with target segment]
Action 2: [specific risk mitigation for financially stressed customers]

OPPORTUNITY
One sentence — the single biggest revenue growth opportunity in the customer base.

Rules: Every finding must cite a number. Be commercially focused."""

    prompt = f"""Write a customer profile briefing based on this aggregated data:

CUSTOMER BASE OVERVIEW:
  Total Customers                 : {kpis['total_customers']:,}
  Average Account Balance         : ${kpis['avg_balance']:,.2f}
  Customers with Negative Balance : {kpis['negative_balance_count']:,} ({kpis['negative_balance_pct']}%)

INSURANCE PENETRATION:
  Customers with Home Insurance   : {kpis['with_home_insurance']:,} ({kpis['home_insurance_pct']}%)
  Customers with Car Loan         : {kpis['with_car_loan']:,} ({kpis['car_loan_pct']}%)

CLAIMS BEHAVIOUR:
  Customers Who Have Filed Claims : {kpis['unique_customers_w_claims']:,}
  Claims Penetration Rate         : {kpis['claims_penetration_pct']}% of customer base

CROSS-SELL CONTEXT:
  Customers WITHOUT Home Insurance: {kpis['total_customers'] - kpis['with_home_insurance']:,}
    → These are prime candidates for home insurance cross-sell
  Customers with Car Loan AND No Claim: ~{kpis['total_customers'] - kpis['unique_customers_w_claims']:,}
    → Low-risk, financially engaged customers
"""

    return get_llm_response(prompt, system=system, max_tokens=700), kpis


print("🤖 Generating Domain 3: Customer Profile insight...")
customer_summary, customer_kpis_used = generate_customer_profile_insight()

print("\n" + "=" * 70)
print("DOMAIN 3 — CUSTOMER PROFILE")
print("=" * 70)
print(customer_summary)
print("=" * 70)

In [0]:
%sql
-- ─────────────────────────────────────────────────────────────
-- CELL 7 — ai_query() Example 1:
-- Generate a plain-English rejection rate summary for each region
-- FIX: Use CTE to pre-aggregate BEFORE calling ai_query()
-- ai_query() runs per-row so aggregate functions cannot be used inside it
-- ─────────────────────────────────────────────────────────────

WITH region_agg AS (
  SELECT
    region,
    ROUND(AVG(rejection_rate_pct), 2) AS avg_rejection_rate_pct,
    SUM(total_claims)                 AS total_claims
  FROM primeinsurance_analytics.gold.agg_claim_rejection_by_region_month
  GROUP BY region
)
SELECT
  region,
  avg_rejection_rate_pct,
  total_claims,
  -- ai_query() now receives scalar values, not aggregate expressions
  get_json_object(
    get_json_object(
      ai_query(
        'databricks-gpt-oss-20b',
        CONCAT(
          'In one sentence, summarise this claims rejection data for an executive: ',
          'Region: ', region,
          ', Average rejection rate: ', avg_rejection_rate_pct, '%',
          ', Total claims: ', total_claims,
          '. State whether this is a concern and what it suggests.'
        )
      ),
      '$[1]'
    ),
    '$.text'
  ) AS ai_regional_summary
FROM region_agg
ORDER BY avg_rejection_rate_pct DESC


In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — ai_query() Example 2:
# Classify each severity tier's processing urgency level
# ─────────────────────────────────────────────────────────────

result = spark.sql("""
SELECT
    incident_severity,
    avg_days_to_process,
    max_days_to_process,
    total_claims,
    get_json_object(
        get_json_object(
            ai_query(
                'databricks-gpt-oss-20b',
                CONCAT(
                    'You are a claims operations analyst. ',
                    'Classify this severity tier as HIGH / MEDIUM / LOW urgency for management attention, ',
                    'then in one sentence explain why. ',
                    'Severity: ', incident_severity,
                    ', Average processing days: ', CAST(avg_days_to_process AS STRING),
                    ', Max processing days: ', CAST(max_days_to_process AS STRING),
                    ', Total claims: ', CAST(total_claims AS STRING),
                    '. Format: URGENCY: [level] — [one sentence reason]'
                )
            ),
            '$[1]'
        ),
        '$.text'
    ) AS ai_urgency_classification
FROM primeinsurance_analytics.gold.agg_claim_processing_time_by_severity
ORDER BY avg_days_to_process DESC
""")

display(result)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 9 — Build Insights Rows and Save to Gold
# ─────────────────────────────────────────────────────────────

insights_rows = [
    # Domain 1 — Policy Portfolio
    {
        "insight_id"           : "INS_001_POLICY_PORTFOLIO",
        "domain_name"          : "policy_portfolio",
        "summary_title"        : "Policy Portfolio Executive Summary",
        "target_audience"      : "Chief Underwriting Officer / Board",
        "ai_generated_summary" : policy_summary,
        "kpi_json"             : json.dumps(policy_kpis_used),
        "model_name"           : MODEL,
        "status"               : "success" if policy_summary else "error"
    },
    # Domain 2 — Claims Performance
    {
        "insight_id"           : "INS_002_CLAIMS_PERFORMANCE",
        "domain_name"          : "claims_performance",
        "summary_title"        : "Claims Performance Executive Summary",
        "target_audience"      : "VP of Claims Operations",
        "ai_generated_summary" : claims_summary,
        "kpi_json"             : json.dumps(claims_kpis_used),
        "model_name"           : MODEL,
        "status"               : "success" if claims_summary else "error"
    },
    # Domain 3 — Customer Profile
    {
        "insight_id"           : "INS_003_CUSTOMER_PROFILE",
        "domain_name"          : "customer_profile",
        "summary_title"        : "Customer Profile Executive Summary",
        "target_audience"      : "Chief Marketing Officer",
        "ai_generated_summary" : customer_summary,
        "kpi_json"             : json.dumps(customer_kpis_used),
        "model_name"           : MODEL,
        "status"               : "success" if customer_summary else "error"
    }
]

# Convert to Spark DataFrame — explicit schema prevents type inference errors
schema = StructType([
    StructField("insight_id",            StringType(), True),
    StructField("domain_name",           StringType(), True),
    StructField("summary_title",         StringType(), True),
    StructField("target_audience",       StringType(), True),
    StructField("ai_generated_summary",  StringType(), True),
    StructField("kpi_json",              StringType(), True),
    StructField("model_name",            StringType(), True),
    StructField("status",                StringType(), True),
])

# Ensure all values are strings (safe None handling)
safe_rows = [
    (
        str(r.get("insight_id", "") or ""),
        str(r.get("domain_name", "") or ""),
        str(r.get("summary_title", "") or ""),
        str(r.get("target_audience", "") or ""),
        str(r.get("ai_generated_summary", "") or ""),
        str(r.get("kpi_json", "") or ""),
        str(r.get("model_name", "") or ""),
        str(r.get("status", "") or ""),
    )
    for r in insights_rows
]
insights_df = spark.createDataFrame(safe_rows, schema=schema)

# Add generated timestamp
insights_df = insights_df.withColumn("generated_at", F.current_timestamp())

# Write to Gold layer (append to preserve history)
(
    insights_df
    .write
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("primeinsurance_analytics.gold.ai_business_insights")
)

print("✅ Saved to primeinsurance_analytics.gold.ai_business_insights")
print(f"   Rows written : {insights_df.count()}")
print(f"   Domains      : policy_portfolio, claims_performance, customer_profile")

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 10 — Verify Output Table
# ─────────────────────────────────────────────────────────────

output = spark.read.table("primeinsurance_analytics.gold.ai_business_insights")

print(f"Total rows in ai_business_insights: {output.count()}")
print("\nSchema:")
output.printSchema()

print("\nInsights generated:")
output.select(
    "insight_id",
    "domain_name",
    "summary_title",
    "model_name",
    "status",
    "generated_at"
).show(truncate=50)

# Verify kpi_json field is populated
print("\nKPI JSON sample (first row):")
first = output.select("insight_id", "kpi_json").collect()[0]
print(f"  {first['insight_id']}:")
print(f"  {first['kpi_json'][:200]}...")

# Print all summaries in full
print("\n" + "=" * 70)
print("FULL INSIGHT PREVIEWS")
print("=" * 70)

for row in output.select("insight_id", "domain_name", "target_audience", "ai_generated_summary").collect():
    print(f"\n── {row['insight_id']} → {row['target_audience']} ──")
    print(row["ai_generated_summary"])
    print()

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 11 — Final Verification: All UC Output Tables
# ─────────────────────────────────────────────────────────────

all_output_tables = [
    "primeinsurance_analytics.gold.dq_explanation_report",
    "primeinsurance_analytics.gold.claim_anomaly_explanations",
    "primeinsurance_analytics.gold.rag_query_history",
    "primeinsurance_analytics.gold.ai_business_insights"
]

print("FINAL VERIFICATION — All UC Output Tables")
print("=" * 60)

for table in all_output_tables:
    try:
        count = spark.read.table(table).count()
        print(f"✅ {table.split('.')[-1]:<40} {count:>6} rows")
    except Exception as e:
        print(f"❌ {table.split('.')[-1]:<40} NOT FOUND — run the corresponding UC notebook")

print("=" * 60)

In [0]:
%sql
select * from primeinsurance_analytics.gold.ai_business_insights